

## Term Project - Projecting Movie Success

In [1]:
# libraries
import pandas as pd
import json
import ast
import re
import requests
import io
import time
import os
import time
import json
import urllib.request
import urllib.parse
from functools import reduce
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

## Data Ingest and Preparation

### TMDB

First I will begin with ingesting and cleaning the TMdB 5000 movies dataset from Kaggle. I have downloaded this file from [Kaggle](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata) and have it in the same directory as my notebook.

In [2]:
# load the TMDb 5000 Movies data into a dataframe
movies = pd.read_csv('tmdb_5000_movies.csv')

# display the first few rows of data
#movies.head()
# get the initial count of movies in the dataset
initial_count = len(movies)

# filter the dataframe to keep only rows with budget AND revenue > 0
tmdb_cleaned = movies[(movies['budget'] > 0) & (movies['revenue'] > 0)].copy()

final_count = len(tmdb_cleaned)
rows_removed = initial_count - final_count

print(f"Rows before: {initial_count}")
print(f"Rows after: {final_count}")
print(f"Total rows removed: {rows_removed}")

# show the first few rows of the updated financial data
#print(tmdb_cleaned[['title', 'budget', 'revenue']].head())

In [56]:
# define the columns I want to keep
cols_to_keep = [
    'title', 'genres', 'budget', 'revenue', 'release_year', 
    'popularity', 'runtime', 'vote_average', 'vote_count'
]

# prune the dataframe
tmdb_cleaned = tmdb_cleaned[cols_to_keep]

# rename headers for better readability
tmdb_cleaned = tmdb_cleaned.rename(columns={
    'title': 'Movie_Title',
    'genres': 'Primary_Genre',
    'budget': 'Budget_USD',
    'revenue': 'Revenue_USD',
    'release_year': 'Release_Year',
    'popularity': 'Popularity_Score',
    'runtime': 'Runtime_Minutes',
    'vote_average': 'Average_Rating',
    'vote_count': 'Total_Votes'
})

#print("New Column Headers:")
#print(tmdb_cleaned.columns.tolist())

In [60]:
# function to normalize the movie titles
def normalize_title(title):
    # convert to lowercase
    title = title.lower()
    # remove special characters (keep only letters and numbers)
    title = re.sub(r'[^a-z0-9\s]', '', title)
    # remove extra internal spaces
    title = re.sub(r'\s+', ' ', title).strip()
    return title

# apply normalization
# create a Simplified_Title column for the stripped version and use the pretty Movie_Title column for displays
tmdb_cleaned['Simplified_Title'] = tmdb_cleaned['Movie_Title'].apply(normalize_title)

#print("Comparison of Original vs. Simplified Title:")
#print(tmdb_cleaned[['Movie_Title', 'Simplified_Title']].head(5))


In [8]:
# length of the dataset after cleaning
print(f"The total length of the cleaned movies dataset is: {len(tmdb_cleaned)}")

# export to CSV
tmdb_cleaned.to_csv('tmdb_cleaned.csv')

The total length of the cleaned movies dataset is: 3229


### OMDb API

OMDb API requires an API key for connection which can be obtained on the [OMDb API site](https://www.omdbapi.com/apikey.aspx)

In [9]:
# setup to reach out to OMDB API
with open('APIkeys.json') as f:
    keys = json.load(f)
omdbapi = keys['OMDBapi']

serviceurl = 'http://www.omdbapi.com/?'
apikey = '&apikey=' + omdbapi

# storage file for fetching titles 1000/day
progress_file = 'omdb_data.csv'

# first day I will need to create the file, subsequent days I want to use the same file
if os.path.exists(progress_file):
    existing = pd.read_csv(progress_file)
    # Get a list of titles we already have
    already_fetched = existing['Movie_Title'].apply(normalize_title).unique().tolist()
    print(f"Resuming: Already have {len(already_fetched)} movies in {progress_file}.")
else:
    existing = pd.DataFrame()
    already_fetched = []
    print("Starting fresh: No previous data found.")

# identify movies that still need to be fetched
all_titles = tmdb_cleaned['Simplified_Title'].unique().tolist()
remaining_titles = [t for t in all_titles if t not in already_fetched]

# limit daily to 975 to stay under my 1000 limit
today_batch = remaining_titles[:975]
print(f"Prepared to fetch {len(today_batch)} new movies today.")

new_records = []

# loop to fetch movies
for movie in today_batch:
    try:
        url = serviceurl + urllib.parse.urlencode({'t': movie}) + apikey
        with urllib.request.urlopen(url) as response:
            data = json.loads(response.read().decode())
            if data.get('Response') == 'True':
                new_records.append(data)
        time.sleep(0.1) # respectful delay
    except Exception as e:
        print(f"Error on {movie}: {e}")

# save and add to dataframe
if new_records:
    new = pd.DataFrame(new_records)
    
    # standardize headers immediately so the append is clean
    new = new.rename(columns={'Title': 'Movie_Title', 'Year': 'Release_Year'})
    
    # combine old data with new data
    updated = pd.concat([existing, new], ignore_index=True)
    
    # save back to CSV for tomorrow
    updated.to_csv(progress_file, index=False)
    print(f"Success! Total records in {progress_file}: {len(updated)}")
else:
    print("No new records were added.")


Resuming: Already have 4481 movies in omdb_data.csv.
Prepared to fetch 239 new movies today.
Success! Total records in omdb_data.csv: 5506


In [57]:
# load the file I created
omdb = pd.read_csv('omdb_data.csv')

# display the first few rows of data
#omdb.head(2)

In [61]:
# pairing down columns and standardizing headers

# list of columns to keep
columns_to_keep = [
    'Movie_Title', 'Release_Year', 'Released', 'Genre', 
    'Awards', 'Ratings', 'Metascore', 'imdbRating', 
    'imdbVotes', 'BoxOffice'
]


# create the new cleaned dataframe by selecting only the specific columns
omdb_cleaned = omdb[columns_to_keep].copy()

# rename specific headers for consistency with my other datasets
omdb_cleaned = omdb_cleaned.rename(columns={
    'imdbRating': 'IMDb_Rating',
    'imdbVotes': 'IMDb_Votes',
    'BoxOffice': 'API_Box_Office',
    'Metascore': 'Metacritic_Score'
})

#omdb_cleaned.head(2)

In [45]:
# Format and Clean Numeric data

# convert Release_Year to a datetime object
# This allows for easier time-based filtering later
omdb_cleaned['Release_Year'] = pd.to_datetime(omdb_cleaned['Release_Year'], errors='coerce')

# convert Metacritic_Score and IMDb_Rating to numeric (floats/ints)
omdb_cleaned['Metacritic_Score'] = pd.to_numeric(omdb_cleaned['Metacritic_Score'], errors='coerce')
omdb_cleaned['IMDb_Rating'] = pd.to_numeric(omdb_cleaned['IMDb_Rating'], errors='coerce')

# clean and convert IMDb_Votes
# remove the commas (e.g., "1,234") before converting to a number
omdb_cleaned['IMDb_Votes'] = pd.to_numeric(
    omdb_cleaned['IMDb_Votes'].str.replace(',', '', regex=False), 
    errors='coerce'
)

# clean and convert API_Box_Office
# strip the '$' and ',' characters
omdb_cleaned['API_Box_Office'] = pd.to_numeric(
    omdb_cleaned['API_Box_Office'].str.replace(r'[\$,]', '', regex=True), 
    errors='coerce'
)

# display info to verify the Dtype changes
#omdb_cleaned.info(2)

In [46]:
# Handle inconsistent values

# fill all missing values (NaN/Null) with the string "N/A"
omdb_cleaned = omdb_cleaned.fillna("N/A")

# calculate the number of "N/A" entries for every column
na_counts = (omdb_cleaned == "N/A").sum()

#print("-" * 30)
#print("Count of 'N/A' values per column:")
#print(na_counts)

In [14]:
# extracting rotten tomatoes scores from the nested "Ratings" strings

def extract_rt(rating_val):
    # If it was already marked as N/A in Step 3, skip it
    if rating_val == "N/A":
        return "N/A"
    
    try:
        # convert the string representation of a list into an actual Python list
        ratings_list = ast.literal_eval(str(rating_val))
        
        for r in ratings_list:
            if r['Source'] == 'Rotten Tomatoes':
                # Strip the '%' and return as a number
                return int(r['Value'].replace('%', ''))
    except (ValueError, SyntaxError):
        return "N/A"
    
    return "N/A"

# apply the function to create new column
omdb_cleaned['RT_Score'] = omdb_cleaned['Ratings'].apply(extract_rt)

# count how many RT scores were found vs how many are N/A
rt_count = (omdb_cleaned['RT_Score'] != "N/A").sum()
rt_na = (omdb_cleaned['RT_Score'] == "N/A").sum()

print(f"Successfully found {rt_count} RT scores. Missing: {rt_na}")

Successfully found 4201 RT scores. Missing: 1305


In [47]:
# function to normalize the movie titles
def normalize_title(title):
    # convert to lowercase
    title = title.lower()
    # remove special characters (keep only letters and numbers)
    title = re.sub(r'[^a-z0-9\s]', '', title)
    # remove extra internal spaces
    title = re.sub(r'\s+', ' ', title).strip()
    return title

# apply normalization
# create a Simplified_Title column for the stripped version and use the pretty Movie_Title column for displays
omdb_cleaned['Simplified_Title'] = omdb_cleaned['Movie_Title'].apply(normalize_title)

#print("*** Comparison of Original vs. Simplified Title:***")
#print(omdb_cleaned[['Movie_Title', 'Simplified_Title']].head(5))

In [49]:
# final review of the cleaned dataset
#print("*** Final Cleaned Dataset Overview: ***")
#display(omdb_cleaned[['Simplified_Title', 'RT_Score', 'Award_Status', 'API_Box_Office']].head(10))

# check the data types one last time
#print("\nFinal Data Types:")
#print(omdb_cleaned.dtypes)

In [50]:
# as I have run and re-run my notebook it continued to fetch new records from OBDb
# this left me with duplicated records, which I do not want and want to clean up
#print(omdb_cleaned.duplicated().sum())
#omdb_cleaned[omdb_cleaned.duplicated()]

In [19]:
# validate I don't have duplicates since every time I run my notebook it queries again for more movies
omdb_cleaned.drop_duplicates(inplace=True)
omdb_cleaned.shape

(4483, 13)

In [20]:
# save to csv
omdb_cleaned.to_csv('omdb_cleaned.csv')

## Combining datasets

In [ ]:
all_movies = pd.merge(
    omdb_cleaned, 
    tmdb_cleaned, 
    on='Simplified_Title', 
    how='inner', 
    suffixes=('_omdb', '_tmdb')
)
#all_movies.head(1)

In [22]:
# creating a csv of my final dataset
all_movies.to_csv('all_movies.csv')

In [59]:
# validating size of dataset
all_movies.shape

(2990, 22)

In [24]:
# checking for missing values
nan_counts = all_movies.isna().sum()
print(nan_counts)

Movie_Title_omdb     0
Release_Year_omdb    0
Released             0
Genre                0
Awards               0
Ratings              0
Metacritic_Score     0
IMDb_Rating          0
IMDb_Votes           0
API_Box_Office       0
RT_Score             0
Award_Status         0
Simplified_Title     0
Movie_Title_tmdb     0
Primary_Genre        0
Budget_USD           0
Revenue_USD          0
Release_Year_tmdb    0
Popularity_Score     0
Runtime_Minutes      0
Average_Rating       0
Total_Votes          0
dtype: int64


### Analysis

In [63]:
## calculate the Pearson correlation between the two revenue streams
# force clean the Box Office column by turning 'N/A' strings into true numeric NaN values
all_movies['API_Box_Office'] = pd.to_numeric(all_movies['API_Box_Office'], errors='coerce')

# REVENUE RECONCILIATION 
print("**** REVENUE RECONCILIATION ****")
# now that it's strictly numeric, the correlation will work 
revenue_corr = all_movies[['Revenue_USD', 'API_Box_Office']].corr().iloc[0, 1]

#print(f"Correlation between TMDB Revenue and OMDB Box Office: {revenue_corr:.4f}")
print(f"Missing values in TMDB 'Revenue_USD': {all_movies['Revenue_USD'].isna().sum()}")
print(f"Missing values in OMDB 'API_Box_Office': {all_movies['API_Box_Office'].isna().sum()}")


**** REVENUE RECONCILIATION ****
Missing values in TMDB 'Revenue_USD': 0
Missing values in OMDB 'API_Box_Office': 178


In [ ]:
#print("**** MISSING CRITIC METRIC SCORES ****")
critic_metrics = ['RT_Score', 'Metacritic_Score', 'IMDb_Rating']

# ensure these are also numeric just in case there are more hidden strings
for col in critic_metrics:
    all_movies[col] = pd.to_numeric(all_movies[col], errors='coerce')

#print(all_movies[critic_metrics].isna().sum())

# create an imputed copy specifically for later use with a baseline Multiple Linear Regression
# decision Trees/Random Forests can deal with missing values later, but regression needs them filled
all_movies_imputed = all_movies.copy()
for col in critic_metrics:
    median_val = all_movies_imputed[col].median()
    all_movies_imputed[f'{col}_Imputed'] = all_movies_imputed[col].fillna(median_val)


In [ ]:
sns.set_theme(style='whitegrid')

# Feature-vs-Target Scatter Plot with Critic Score gradient
plt.figure(figsize=(10, 6))
# Filter out rows with missing RT_Score for this visualization
scatter_df = all_movies.dropna(subset=['RT_Score', 'Budget_USD', 'Revenue_USD'])
scatter = plt.scatter(
    scatter_df['Budget_USD'] / 1e6, 
    scatter_df['Revenue_USD'] / 1e6, 
    c=scatter_df['RT_Score'], 
    cmap='viridis', 
    alpha=0.7, 
    edgecolors='w', 
    linewidth=0.5
)
cbar = plt.colorbar(scatter)
cbar.set_label('Rotten Tomatoes Score (%)', fontsize=11)
plt.title('Movie Revenue vs. Production Budget\n(Color-coded by Rotten Tomatoes Score)', fontsize=14, fontweight='bold')
plt.xlabel('Budget (Millions USD)', fontsize=12)
plt.ylabel('Revenue (Millions USD)', fontsize=12)
plt.tight_layout()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 8))
corr_cols = ['Budget_USD', 'Revenue_USD', 'IMDb_Rating', 'RT_Score', 'Metacritic_Score', 'Popularity_Score', 'Total_Votes']
corr_matrix = all_movies[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Continuous Features', fontsize=14, fontweight='bold')
plt.tight_layout()

In [ ]:
# Target Variable Distribution Check

# filter out zero or negative values to avoid mathematical errors with log scales
regression_check = all_movies[(all_movies['Budget_USD'] > 0) & (all_movies['Revenue_USD'] > 0)].dropna(subset=['RT_Score'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# raw revenue distribution
sns.histplot(regression_check['Revenue_USD'] / 1e6, bins=30, kde=True, ax=axes[0], color='blue')
axes[0].set_title('Raw Revenue Distribution (Millions USD)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Revenue (Millions USD)')
axes[0].set_ylabel('Count')

# log-transformed revenue distribution
sns.histplot(np.log10(regression_check['Revenue_USD']), bins=30, kde=True, ax=axes[1], color='green')
axes[1].set_title(r'Log-Transformed Revenue Distribution ($\log_{10}$)', fontsize=12, fontweight='bold')
axes[1].set_xlabel(r'$\log_{10}$(Revenue_USD)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Linearity Check (Feature vs. Target)

plt.figure(figsize=(9, 5))

# using a 2nd-order polynomial fit (order=2) to spot bends/thresholds
sns.regplot(
    data=regression_check,
    x='RT_Score', 
    y=np.log10(regression_check['Revenue_USD']), 
    scatter_kws={'alpha': 0.25, 'color': 'gray'}, 
    line_kws={'color': 'red', 'linewidth': 2}, 
    order=2 
)

plt.title(r'Log Revenue vs. Rotten Tomatoes Score', fontsize=12, fontweight='bold')
plt.xlabel('Rotten Tomatoes Score (%)')
plt.ylabel(r'$\log_{10}$(Revenue_USD)')
plt.tight_layout()
plt.show()

## Feature Engineering

I will create a new feature called `Combined_Score` that will sum the RT_Score and the Metacritic_Score and then average.

$Combined\_Score = \frac{Metacritic\_Score + RT\_Score}{2}$



In [36]:
# flter out zero/negative financial values
model = all_movies[(all_movies['Budget_USD'] > 0) & (all_movies['Revenue_USD'] > 0)].copy()

# convert to numeric dtypes
numeric_cols = ['RT_Score', 'Metacritic_Score', 'IMDb_Rating', 'Popularity_Score', 'Total_Votes', 'Budget_USD', 'Revenue_USD']
for col in numeric_cols:
    model[col] = pd.to_numeric(model[col], errors='coerce')

# impute missing values
for col in ['RT_Score', 'Metacritic_Score', 'IMDb_Rating']:
    model[col] = model[col].fillna(model[col].median())


# FEATURE ENGINEERING: Create Combined_Score

model['Combined_Score'] = (model['RT_Score'] + model['Metacritic_Score']) / 2.0

# updated feature set using the engineered column
features = ['Budget_USD', 'Combined_Score', 'IMDb_Rating', 'Popularity_Score', 'Total_Votes']



# Modeling

## Multiple Linear Regression

In [37]:

# MULTIPLE LINEAR REGRESSION

# use the features I made above after my feature engineering
X_lr = model[features]
y_lr_log = np.log10(model['Revenue_USD'])

# 80/20 train/test split
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X_lr, y_lr_log, test_size=0.2, random_state=42)

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)

# calculate log-scale residuals on the TRAINING set
train_log_preds = lr_model.predict(X_train_lr)
train_residuals_log = y_train_lr - train_log_preds

# compute Duan's Smearing Factor for Base-10 Log
duan_smearing_factor = np.mean(10 ** train_residuals_log)

# generate test predictions (Log scale)
y_pred_log = lr_model.predict(X_test_lr)

# apply back-transformation with smearing factor correction
y_test_usd = 10 ** y_test_lr
y_pred_usd_smeared = (10 ** y_pred_log) * duan_smearing_factor

# evaluate performance on original scale
# R2 computed on LOG scale; this is standard practice due to the way Ordinary Least Squares optimization & back-transformation works
r2_lr = r2_score(y_test_lr, y_pred_log)  
mae_lr = mean_absolute_error(y_test_usd, y_pred_usd_smeared)
rmse_lr = np.sqrt(mean_squared_error(y_test_usd, y_pred_usd_smeared))

print(f"NOTE: Log values have been inverse-transformed and smearing correction applied.")
print(f"Duan's Smearing Factor (phi): {duan_smearing_factor:.4f}")

print(f"Linear Regression -> R\u00B2: {r2_lr:.4f} | MAE: ${mae_lr/1e6:.2f}M | RMSE: ${rmse_lr/1e6:.2f}M")


NOTE: Log values have been inverse-transformed and smearing correction applied.
Duan's Smearing Factor (phi): 1.9661
Linear Regression -> R²: 0.3802 | MAE: $525.26M | RMSE: $5896.79M


## Optimized Decision Tree Regressor

In [38]:

# OPTIMIZED DECISION TREE REGRESSOR
# use my same features
X_dt = model[features]
# target
y_dt = model['Revenue_USD']  

# same 80/20 test split
X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(X_dt, y_dt, test_size=0.2, random_state=42)

# hyperparemeter search grid: this setup will enable 15 variations of the model
# max_depth=how tall or deep a single decision tree can grow; shallow, moderate, and unconstrained
# min_samples_split= minimum number of data row required in a node before tree is allowed to split again; default and higher thresholds
param_grid = {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]}
dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=42), param_grid, cv=5, scoring='r2')
dt_grid.fit(X_train_dt, y_train_dt)

# return the actual fully-trained model object that had the highest cross-validation score
best_dt = dt_grid.best_estimator_
y_pred_dt = best_dt.predict(X_test_dt)

r2_dt = r2_score(y_test_dt, y_pred_dt)
mae_dt = mean_absolute_error(y_test_dt, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test_dt, y_pred_dt))

print("**** OPTIMIZED DECISION TREE REGRESSOR PERFORMANCE ****")
print(f"Decision Tree Regressor -> R\u00B2: {r2_dt:.4f} | MAE: ${mae_dt/1e6:.2f}M | RMSE: ${rmse_dt/1e6:.2f}M")


**** OPTIMIZED DECISION TREE REGRESSOR PERFORMANCE ****
Decision Tree Regressor -> R²: 0.6610 | MAE: $64.58M | RMSE: $122.28M


## Random Forest Regressor

In [39]:

# RANDOM FOREST REGRESSOR
# same features
X_rf = model[features]
# target raw revenue
y_rf = model['Revenue_USD']

# same 80/20 test split
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=42)
rf_model.fit(X_train_rf, y_train_rf)

y_pred_rf = rf_model.predict(X_test_rf)

r2_rf = r2_score(y_test_rf, y_pred_rf)
mae_rf = mean_absolute_error(y_test_rf, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test_rf, y_pred_rf))

print(f"Random Forest Regressor -> R\u00B2: {r2_rf:.4f} | MAE: ${mae_rf/1e6:.2f}M | RMSE: ${rmse_rf/1e6:.2f}M")



Random Forest Regressor -> R²: 0.7252 | MAE: $57.90M | RMSE: $110.10M


In [51]:
# extract feature importances from random forest
importances = rf_model.feature_importances_
feature_names = X_rf.columns

# creating a clean dataframe for review
feature_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

#print("**** RANDOM FOREST FEATURE IMPORTANCES ****")
#print(feature_importances.to_string(index=False))

| Feature | Importance | Takeaway | 
| :--- | :--- | :--- |
| Total Votes | $58.4\%$ | The primary driver of a movie's success based on votes on movie platforms like IMDb; strong indicator of audience engagement and volume of viewers. More than half the model's predictive power is from this single variable |
| Budget | $29.6\%$ | Production budget accounts for just under a third of determining success of a movie. A higher budget will allow for more marketing, better, and more, theaters for release, and bigger movie stars |
| Popularity | $5.6\%$ | This is a very small driver of success and is typically around a movie's release |
| Combined Score | $3.6\%$ | The average score of Rotten Tomatoes and the metacritic score. These play a minor role in predicting movie success |
| IMDb Rating | $2.8\%$ | Ratings on IMDb are not playing a large role in the overall success of a movie, but helps to provide input for any edge cases such as a sleeper hit or an overhyped flop |
Table: Random Forest Feature Importances


### Tuning & Optimizations for Random Forest Regressor

In [44]:
# RANDOM FOREST REGRESSOR TUNING & OPTIMIZATION

# new dataframe- ensures no data leakage
model = all_movies[(all_movies['Budget_USD'] > 0) & (all_movies['Revenue_USD'] > 0)].copy()

raw_features = ['Budget_USD', 'RT_Score', 'Metacritic_Score', 'IMDb_Rating', 'Popularity_Score', 'Total_Votes']
X = model[raw_features].copy()
y = model['Revenue_USD'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# FEATURE ENGINEERING - ONLY TRAINING DATA
def engineer_rf_features(X_tr_df, X_te_df):
    X_tr = X_tr_df.copy()
    X_te = X_te_df.copy()
    
    # impute missing values using training medians
    imputer = SimpleImputer(strategy='median')
    critic_cols = ['RT_Score', 'Metacritic_Score', 'IMDb_Rating']
    
    X_tr[critic_cols] = imputer.fit_transform(X_tr[critic_cols])
    X_te[critic_cols] = imputer.transform(X_te[critic_cols])
    
    # FEATURE ENGINEERING - Combined Critic Score
    X_tr['Combined_Score'] = (X_tr['RT_Score'] + X_tr['Metacritic_Score']) / 2.0
    X_te['Combined_Score'] = (X_te['RT_Score'] + X_te['Metacritic_Score']) / 2.0
    
    # FEATURE ENGINEERING- Critic vs User Sentiment Gap
    X_tr['Critic_User_Diff'] = X_tr['Combined_Score'] - (X_tr['IMDb_Rating'] * 10.0)
    X_te['Critic_User_Diff'] = X_te['Combined_Score'] - (X_te['IMDb_Rating'] * 10.0)
    
    # LOG TRANSFORM Skewed Predictors
    X_tr['Log_Budget'] = np.log10(X_tr['Budget_USD'])
    X_te['Log_Budget'] = np.log10(X_te['Budget_USD'])
    
    X_tr['Log_Votes'] = np.log10(X_tr['Total_Votes'] + 1)
    X_te['Log_Votes'] = np.log10(X_te['Total_Votes'] + 1)
    
    # FEATURE ENGINEERING- Budget per Vote Ratio
    X_tr['Budget_Per_Vote'] = X_tr['Budget_USD'] / (X_tr['Total_Votes'] + 1)
    X_te['Budget_Per_Vote'] = X_te['Budget_USD'] / (X_te['Total_Votes'] + 1)
    
    cols = ['Log_Budget', 'Log_Votes', 'Combined_Score', 'IMDb_Rating', 
            'Popularity_Score', 'Critic_User_Diff', 'Budget_Per_Vote']
    
    return X_tr[cols], X_te[cols]

X_train_eng, X_test_eng = engineer_rf_features(X_train, X_test)

# ADVANCED RANDOM FOREST GRID SEARCH
# log-transform target for training to handle target skewness
y_train_log = np.log10(y_train)

# hyperparameter grid definitions
param_grid_rf = {
    # number of trees
    'n_estimators': [300, 500],
    # depth of trees
    'max_depth': [10, 15, 20, None],
    # how many features per node;
    'max_features': [0.4, 0.5, 0.6, 0.7],
    # minimum number of data rows before a node can split
    'min_samples_split': [2, 4, 6],
    # minimum number of data rows in a final leaf node (should smooth out extreme estimates)
    'min_samples_leaf': [1, 2]
}

# cross-validation grid search
rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_rf,
    # 5 fold cross validation
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# runs the 192 combinations (param_grid) x 5 CV folds = 960 total model fits
rf_grid.fit(X_train_eng, y_train_log)
# extract the top performer from the grid search for the final test set eval
best_rf = rf_grid.best_estimator_

# compute Duan's Smearing Factor on log residuals
train_log_preds = best_rf.predict(X_train_eng)
train_res_log = y_train_log - train_log_preds
duan_smearing = np.mean(10 ** train_res_log)

# EVALUATION - on hold-out TEST DATA
test_log_preds = best_rf.predict(X_test_eng)
y_pred_usd_smeared = (10 ** test_log_preds) * duan_smearing

o_r2_rf = r2_score(y_test, y_pred_usd_smeared)
o_mae_rf = mean_absolute_error(y_test, y_pred_usd_smeared)
o_rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_usd_smeared))

print("**** OPTIMIZED RANDOM FOREST PERFORMANCE ****")
print(f"Optimal Parameters: {rf_grid.best_params_}")
print(f"Duan Smearing Factor: {duan_smearing:.4f}")
print(f"Optimized Random Forest Regressor -> R\u00B2: {o_r2_rf:.4f} | MAE: ${o_mae_rf/1e6:.2f}M | RMSE: ${o_rmse_rf/1e6:.2f}M")


**** OPTIMIZED RANDOM FOREST PERFORMANCE ****
Optimal Parameters: {'max_depth': 10, 'max_features': 0.4, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
Duan Smearing Factor: 1.2965
Optimized Random Forest Regressor -> R²: 0.7393 | MAE: $58.54M | RMSE: $107.25M


In [ ]:

# extract feature importances from the best estimator
features = best_rf.feature_importances_
feature_names = X_train_eng.columns

# create DataFrame sorted by importance
feature_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance_Raw': features,
    'Importance_Pct': features * 100
}).sort_values(by='Importance_Pct', ascending=False).reset_index(drop=True)

# tabular breakdown
#print("**** FEATURE IMPORTANCE BREAKDOWN ****")
#for idx, row in feature_importances.iterrows():
  #  print(f"{row['Feature']:<20} | {row['Importance_Pct']:6.2f}%")

# plotting
plt.figure(figsize=(8, 5))
sns.barplot(
    data=feature_importances, 
    x='Importance_Pct', 
    y='Feature', 
    hue='Feature',
    legend=False,
    palette='viridis'
)

plt.title('Optimized Random Forest Feature Importance (%)', fontsize=12, fontweight='bold')
plt.xlabel('Importance Percentage (%)')
plt.ylabel('Predictor Feature')
plt.xlim(0, max(feature_importances['Importance_Pct']) + 5)

# add percentage annotations to bars
for index, value in enumerate(feature_importances['Importance_Pct']):
    plt.text(value + 0.5, index, f"{value:.2f}%", va='center', fontsize=10)

plt.tight_layout()
plt.show()

## Final Model Comparison and Summary

**Final Model Comparison**

| Model Type | R-Squared | MAE | RMSE |
| :--- | :--- | :--- | :--- |
| Multiple Linear Regression | $38.02\%$ | $\$525.26M$ | $\$5,896.79M$ |
| Optimized Decision Tree | $66.10\%$ | $\$64.58M$ | $\$121.28M$ |
| Random Forest Regressor | $72.52\%$ | $\$57.90M$ | $\$110.10M$ |
| Optimized Random Forest Regressor | $73.93\%$ | $\$58.54M$ | $\$107.25M$ | 
Table: Final Model Comparison